In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout,Input
from tensorflow.keras.optimizers import Adam
import numpy as np
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from pathlib import Path
import matplotlib.pyplot as plt

from tensorflow.keras.models import load_model
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,    
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)


In [ ]:
# ============================================================
# 1. LOAD SAVED LSTM SEQUENCES
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
SEQUENCE_DIR = Path("../data/sequences")

# ------------------------------------------------------------
# Load sequences
# ------------------------------------------------------------
X_train = np.load(SEQUENCE_DIR / "train_lstm_sequences_X.npy")
y_train = np.load(SEQUENCE_DIR / "train_lstm_sequences_y.npy")

X_validation = np.load(SEQUENCE_DIR / "validation_lstm_sequences_X.npy")
y_validation = np.load(SEQUENCE_DIR / "validation_lstm_sequences_y.npy")

X_test = np.load(SEQUENCE_DIR / "test_lstm_sequences_X.npy")
y_test = np.load(SEQUENCE_DIR / "test_lstm_sequences_y.npy")

print("Sequences loaded successfully.")

## Verify the loaded data

In [ ]:

# ------------------------------------------------------------
# Expected configuration
# ------------------------------------------------------------
EXPECTED_SEQUENCE_LENGTH = 20
EXPECTED_N_FEATURES = 19

print("=" * 70)
print("LSTM SEQUENCE VERIFICATION")
print("=" * 70)


# ============================================================
# 1. CHECK SHAPES
# ============================================================

print("\n" + "-" * 70)
print("1. SHAPES")
print("-" * 70)

print(f"X_train:       {X_train.shape}")
print(f"y_train:       {y_train.shape}")

print(f"X_validation:  {X_validation.shape}")
print(f"y_validation:  {y_validation.shape}")

print(f"X_test:        {X_test.shape}")
print(f"y_test:        {y_test.shape}")

In [ ]:
# ============================================================
# 2. CHECK SEQUENCE LENGTH AND NUMBER OF FEATURES
# ============================================================

print("\n" + "-" * 70)
print("2. SEQUENCE CONFIGURATION")
print("-" * 70)

assert X_train.ndim == 3, "X_train must be 3-dimensional"
assert X_validation.ndim == 3, "X_validation must be 3-dimensional"
assert X_test.ndim == 3, "X_test must be 3-dimensional"

assert X_train.shape[1] == EXPECTED_SEQUENCE_LENGTH
assert X_validation.shape[1] == EXPECTED_SEQUENCE_LENGTH
assert X_test.shape[1] == EXPECTED_SEQUENCE_LENGTH

assert X_train.shape[2] == EXPECTED_N_FEATURES
assert X_validation.shape[2] == EXPECTED_N_FEATURES
assert X_test.shape[2] == EXPECTED_N_FEATURES

print(f"Sequence length: {X_train.shape[1]}")
print(f"Number of features: {X_train.shape[2]}")

print("✓ Sequence length = 20")
print("✓ Number of features = 19")

In [ ]:
print("\n" + "-" * 70)
print("3. X / Y SAMPLE COUNTS")
print("-" * 70)

assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

print(f"Train:      {len(X_train):,} sequences / {len(y_train):,} labels")
print(f"Validation: {len(X_validation):,} sequences / {len(y_validation):,} labels")
print(f"Test:       {len(X_test):,} sequences / {len(y_test):,} labels")

print("✓ X and y lengths match for all datasets")

In [ ]:
# ============================================================
# 4. CHECK NaN VALUES
# ============================================================

print("\n" + "-" * 70)
print("4. NaN CHECK")
print("-" * 70)

train_nan = np.isnan(X_train).sum()
validation_nan = np.isnan(X_validation).sum()
test_nan = np.isnan(X_test).sum()

print(f"Train NaNs:      {train_nan:,}")
print(f"Validation NaNs: {validation_nan:,}")
print(f"Test NaNs:       {test_nan:,}")

assert train_nan == 0, "NaNs found in X_train"
assert validation_nan == 0, "NaNs found in X_validation"
assert test_nan == 0, "NaNs found in X_test"

print("✓ No NaN values")

In [ ]:
print("\n" + "-" * 70)
print("5. INFINITE VALUE CHECK")
print("-" * 70)

train_inf = np.isinf(X_train).sum()
validation_inf = np.isinf(X_validation).sum()
test_inf = np.isinf(X_test).sum()

print(f"Train infinities:      {train_inf:,}")
print(f"Validation infinities: {validation_inf:,}")
print(f"Test infinities:       {test_inf:,}")

assert train_inf == 0, "Infinite values found in X_train"
assert validation_inf == 0, "Infinite values found in X_validation"
assert test_inf == 0, "Infinite values found in X_test"

print("✓ No infinite values")

In [ ]:
# ============================================================
# 6. CHECK TARGET VALUES
# ============================================================

print("\n" + "-" * 70)
print("6. TARGET VALUES")
print("-" * 70)

for name, y in [
    ("TRAIN", y_train),
    ("VALIDATION", y_validation),
    ("TEST", y_test)
]:
    
    unique_values = np.unique(y)
    
    print(f"\n{name}")
    print(f"Unique values: {unique_values}")
    
    assert set(unique_values).issubset({0, 1}), (
        f"{name} contains values other than 0 and 1"
    )

print("\n✓ All targets contain only 0 and 1")

In [ ]:
# ============================================================
# 7. CLASS DISTRIBUTION
# ============================================================

print("\n" + "-" * 70)
print("7. CLASS DISTRIBUTION")
print("-" * 70)

for name, y in [
    ("TRAIN", y_train),
    ("VALIDATION", y_validation),
    ("TEST", y_test)
]:
    
    values, counts = np.unique(y, return_counts=True)
    
    print(f"\n{name}")
    
    for value, count in zip(values, counts):
        percentage = count / len(y) * 100
        
        label = "Normal" if value == 0 else "Anomaly"
        
        print(
            f"  Class {value} ({label}): "
            f"{count:,} "
            f"({percentage:.2f}%)"
        )

In [ ]:
# ============================================================
# 8. CHECK TARGET DATA TYPES
# ============================================================

print("\n" + "-" * 70)
print("8. DATA TYPES")
print("-" * 70)

print(f"X_train dtype:       {X_train.dtype}")
print(f"X_validation dtype:  {X_validation.dtype}")
print(f"X_test dtype:        {X_test.dtype}")

print(f"y_train dtype:       {y_train.dtype}")
print(f"y_validation dtype:  {y_validation.dtype}")
print(f"y_test dtype:        {y_test.dtype}")

In [ ]:
# ============================================================
# 3. CALCULATE CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

print("=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

# ------------------------------------------------------------
# Get classes from TRAIN only
# ------------------------------------------------------------
classes = np.unique(y_train)

print(f"\nTraining classes: {classes}")

# ------------------------------------------------------------
# Calculate balanced class weights
# ------------------------------------------------------------
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

# Convert to dictionary for Keras
class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights_array)
}

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------
print("\nClass weights:")

for cls, weight in class_weights.items():
    label = "Normal" if cls == 0 else "Anomaly"
    print(
        f"  Class {cls} ({label}): "
        f"{weight:.4f}"
    )

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------
assert set(class_weights.keys()) == {0, 1}

print("\n✓ Class weights calculated from TRAIN only")
print("✓ Ready for LSTM training")

## Model Training

In [ ]:
print("=" * 70)
print("BUILDING LSTM CLASSIFIER")
print("=" * 70)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------------------
# Input configuration
# ------------------------------------------------------------
SEQUENCE_LENGTH = X_train.shape[1]
N_FEATURES = X_train.shape[2]

print(f"\nInput sequence length: {SEQUENCE_LENGTH}")
print(f"Number of features:   {N_FEATURES}")


In [ ]:
model = Sequential([
    
    # Explicit input layer
    Input(shape=(SEQUENCE_LENGTH, N_FEATURES)),
    
    # First LSTM layer
    LSTM(
        64,
        return_sequences=True
    ),
    
    Dropout(0.2),
    
    # Second LSTM layer
    LSTM(
        32,
        return_sequences=False
    ),
    
    Dropout(0.2),
    
    # Dense classification layer
    Dense(
        16,
        activation="relu"
    ),
    
    # Binary classification output
    Dense(
        1,
        activation="sigmoid"
    )
])

In [ ]:
# ------------------------------------------------------------
# Display architecture
# ------------------------------------------------------------
model.summary()

print("\n✓ LSTM model created successfully")
print("✓ Input shape:", (SEQUENCE_LENGTH, N_FEATURES))

In [ ]:
# ============================================================
# 5. COMPILE LSTM CLASSIFIER
# ============================================================
print("=" * 70)
print("COMPILING LSTM CLASSIFIER")
print("=" * 70)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

print("\n✓ Model compiled successfully")
print("\nLoss: Binary Crossentropy")
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Metrics: Accuracy, Precision, Recall, ROC-AUC, PR-AUC")

In [ ]:
# ============================================================
# 6. TRAIN LSTM CLASSIFIER
# ============================================================

print("=" * 70)
print("LSTM TRAINING")
print("=" * 70)


# ============================================================
# 1. MODEL OUTPUT DIRECTORY
# ============================================================

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "lstm_classifier_best.keras"


# ============================================================
# 2. CALLBACKS
# ============================================================

# ------------------------------------------------------------
# Early stopping
# ------------------------------------------------------------
early_stopping = EarlyStopping(
    monitor="val_pr_auc",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# ------------------------------------------------------------
# Save best model
# ------------------------------------------------------------
model_checkpoint = ModelCheckpoint(
    filepath=BEST_MODEL_PATH,
    monitor="val_pr_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

# ------------------------------------------------------------
# Reduce learning rate when validation performance stops
# improving
# ------------------------------------------------------------
reduce_lr = ReduceLROnPlateau(
    monitor="val_pr_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)


# ============================================================
# 3. TRAINING CONFIGURATION
# ============================================================

EPOCHS = 50
BATCH_SIZE = 256

print("\nTraining configuration:")
print(f"Epochs:       {EPOCHS}")
print(f"Batch size:   {BATCH_SIZE}")
print(f"Class weights: {class_weights}")

print("\nCallbacks:")
print("✓ EarlyStopping")
print("✓ ModelCheckpoint")
print("✓ ReduceLROnPlateau")


# ============================================================
# 4. TRAIN MODEL
# ============================================================

history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_validation,
        y_validation
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,

    class_weight=class_weights,

    callbacks=[
        early_stopping,
        model_checkpoint,
        reduce_lr
    ],

    verbose=1
)


# ============================================================
# 5. TRAINING SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(f"Epochs actually trained: {len(history.history['loss'])}")
print(f"Best model saved to: {BEST_MODEL_PATH}")

print("\n✓ Training completed")
print("✓ Best validation PR-AUC model restored")

## ANALYZE LSTM TRAINING HISTORY


In [ ]:
print("=" * 70)
print("LSTM TRAINING HISTORY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# Extract history
# ------------------------------------------------------------

history_dict = history.history

epochs = range(1, len(history_dict["loss"]) + 1)

# ------------------------------------------------------------
# Best validation epochs
# ------------------------------------------------------------

best_pr_auc_epoch = (
    np.argmax(history_dict["val_pr_auc"]) + 1
)

best_roc_auc_epoch = (
    np.argmax(history_dict["val_roc_auc"]) + 1
)

best_val_loss_epoch = (
    np.argmin(history_dict["val_loss"]) + 1
)

print("\nBEST VALIDATION RESULTS")
print("-" * 70)

print(
    f"Best validation PR-AUC: "
    f"{max(history_dict['val_pr_auc']):.4f} "
    f"(epoch {best_pr_auc_epoch})"
)

print(
    f"Best validation ROC-AUC: "
    f"{max(history_dict['val_roc_auc']):.4f} "
    f"(epoch {best_roc_auc_epoch})"
)

print(
    f"Lowest validation loss: "
    f"{min(history_dict['val_loss']):.4f} "
    f"(epoch {best_val_loss_epoch})"
)


In [ ]:
# ------------------------------------------------------------
# Final epoch
# ------------------------------------------------------------

final_epoch = len(history_dict["loss"])

print("\nFINAL TRAINING EPOCH")
print("-" * 70)

print(f"Final epoch: {final_epoch}")
print(f"Training loss: {history_dict['loss'][-1]:.4f}")
print(f"Validation loss: {history_dict['val_loss'][-1]:.4f}")

print(f"Training PR-AUC: {history_dict['pr_auc'][-1]:.4f}")
print(f"Validation PR-AUC: {history_dict['val_pr_auc'][-1]:.4f}")

print(f"Training ROC-AUC: {history_dict['roc_auc'][-1]:.4f}")
print(f"Validation ROC-AUC: {history_dict['val_roc_auc'][-1]:.4f}")


In [ ]:
# ============================================================
# PLOT 1 — LOSS
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history_dict["loss"],
    label="Training Loss"
)

plt.plot(
    epochs,
    history_dict["val_loss"],
    label="Validation Loss"
)

plt.axvline(
    best_pr_auc_epoch,
    linestyle="--",
    label=f"Best PR-AUC (epoch {best_pr_auc_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("LSTM Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# PLOT 2 — PR-AUC
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history_dict["pr_auc"],
    label="Training PR-AUC"
)

plt.plot(
    epochs,
    history_dict["val_pr_auc"],
    label="Validation PR-AUC"
)

plt.axvline(
    best_pr_auc_epoch,
    linestyle="--",
    label=f"Best Validation PR-AUC (epoch {best_pr_auc_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("PR-AUC")
plt.title("LSTM Training vs Validation PR-AUC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# PLOT 3 — ROC-AUC
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history_dict["roc_auc"],
    label="Training ROC-AUC"
)

plt.plot(
    epochs,
    history_dict["val_roc_auc"],
    label="Validation ROC-AUC"
)

plt.axvline(
    best_roc_auc_epoch,
    linestyle="--",
    label=f"Best Validation ROC-AUC (epoch {best_roc_auc_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("ROC-AUC")
plt.title("LSTM Training vs Validation ROC-AUC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# PLOT 4 — PRECISION
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history_dict["precision"],
    label="Training Precision"
)

plt.plot(
    epochs,
    history_dict["val_precision"],
    label="Validation Precision"
)

plt.xlabel("Epoch")
plt.ylabel("Precision")
plt.title("LSTM Training vs Validation Precision")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# PLOT 5 — RECALL
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    history_dict["recall"],
    label="Training Recall"
)

plt.plot(
    epochs,
    history_dict["val_recall"],
    label="Validation Recall"
)

plt.xlabel("Epoch")
plt.ylabel("Recall")
plt.title("LSTM Training vs Validation Recall")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TRAINING HISTORY SUMMARY")
print("=" * 70)

print(
    f"✓ Best validation PR-AUC: "
    f"{max(history_dict['val_pr_auc']):.4f} "
    f"at epoch {best_pr_auc_epoch}"
)

print(
    f"✓ Best validation ROC-AUC: "
    f"{max(history_dict['val_roc_auc']):.4f} "
    f"at epoch {best_roc_auc_epoch}"
)

print(
    f"✓ Lowest validation loss: "
    f"{min(history_dict['val_loss']):.4f} "
    f"at epoch {best_val_loss_epoch}"
)

if best_pr_auc_epoch < final_epoch:
    print(
        "\n⚠ Validation PR-AUC peaked before the final epoch."
    )
    print(
        "  This indicates that continued training did not improve "
        "validation PR-AUC."
    )
else:
    print(
        "\n✓ Validation PR-AUC was still improving at the final epoch."
    )

## Threshold selection using validation only

In [ ]:
# ==============================================================
# STEP 8 — VALIDATION PROBABILITIES
# ==============================================================
MODEL_PATH = "../models/lstm_classifier_best.keras"

# Load the best model selected during training
best_model = load_model(MODEL_PATH)

print("=" * 70)
print("BEST LSTM MODEL LOADED")
print("=" * 70)

print(f"Model path: {MODEL_PATH}")
print(f"Validation samples: {len(X_validation):,}")

# Predict anomaly probabilities
y_validation_proba = best_model.predict(
    X_validation,
    batch_size=256,
    verbose=1
).ravel()

print("\nProbability summary")
print("-" * 70)
print(f"Minimum probability: {y_validation_proba.min():.6f}")
print(f"Maximum probability: {y_validation_proba.max():.6f}")
print(f"Mean probability:    {y_validation_proba.mean():.6f}")
print(f"Median probability:  {np.median(y_validation_proba):.6f}")

print("\n✓ Validation probabilities generated.")

In [ ]:
# ==============================================================
# STEP 9 — VALIDATION THRESHOLD ANALYSIS
# ==============================================================

# Candidate thresholds
thresholds = np.arange(0.05, 1.00, 0.05)

results = []

for threshold in thresholds:

    y_pred = (y_validation_proba >= threshold).astype(np.int8)

    precision = precision_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        y_pred,
        labels=[0, 1]
    ).ravel()

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })

threshold_results = pd.DataFrame(results)

print("=" * 90)
print("VALIDATION THRESHOLD ANALYSIS")
print("=" * 90)

print(threshold_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# ==============================================================
# BEST F1 THRESHOLD
# ==============================================================

best_f1_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

print("\n" + "=" * 70)
print("BEST VALIDATION F1 THRESHOLD")
print("=" * 70)

print(f"Threshold : {best_f1_row['threshold']:.2f}")
print(f"Precision : {best_f1_row['precision']:.4f}")
print(f"Recall    : {best_f1_row['recall']:.4f}")
print(f"F1-score  : {best_f1_row['f1']:.4f}")

print("\nConfusion matrix components")
print("-" * 70)
print(f"TN: {int(best_f1_row['true_negatives']):,}")
print(f"FP: {int(best_f1_row['false_positives']):,}")
print(f"FN: {int(best_f1_row['false_negatives']):,}")
print(f"TP: {int(best_f1_row['true_positives']):,}")

In [ ]:
# ==============================================================
# STEP 10 — FINE VALIDATION THRESHOLD SEARCH
# ==============================================================

fine_thresholds = np.arange(0.20, 0.301, 0.01)

fine_results = []

for threshold in fine_thresholds:

    y_pred = (y_validation_proba >= threshold).astype(np.int8)

    precision = precision_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        y_pred,
        labels=[0, 1]
    ).ravel()

    fine_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

fine_threshold_results = pd.DataFrame(fine_results)

best_fine = fine_threshold_results.loc[
    fine_threshold_results["f1"].idxmax()
]

print("=" * 80)
print("FINE VALIDATION THRESHOLD SEARCH")
print("=" * 80)

print(
    fine_threshold_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 70)
print("SELECTED VALIDATION THRESHOLD")
print("=" * 70)

print(f"Threshold : {best_fine['threshold']:.2f}")
print(f"Precision : {best_fine['precision']:.4f}")
print(f"Recall    : {best_fine['recall']:.4f}")
print(f"F1-score  : {best_fine['f1']:.4f}")

print("\nConfusion matrix components")
print("-" * 70)
print(f"TN: {int(best_fine['TN']):,}")
print(f"FP: {int(best_fine['FP']):,}")
print(f"FN: {int(best_fine['FN']):,}")
print(f"TP: {int(best_fine['TP']):,}")

### The search confirms the threshold should be 0.24 if we use validation F1 as the selection criterion.

# TEST EVALUATION

In [ ]:
FINAL_THRESHOLD = 0.24

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)

y_test_proba = best_model.predict(
    X_test,
    batch_size=256,
    verbose=1
).ravel()

print("\n✓ Test probabilities generated.")

In [ ]:
y_test_pred = (
    y_test_proba >= FINAL_THRESHOLD
).astype(np.int8)

# --------------------------------------------------------------
# 3. Threshold-independent metrics
# --------------------------------------------------------------

test_roc_auc = roc_auc_score(
    y_test,
    y_test_proba
)

test_pr_auc = average_precision_score(
    y_test,
    y_test_proba
)

In [ ]:
# --------------------------------------------------------------
# 4. Threshold-dependent metrics
# --------------------------------------------------------------

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    zero_division=0
)

In [ ]:
tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1]
).ravel()

In [ ]:
print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Decision threshold : {FINAL_THRESHOLD:.2f}")

print("\nThreshold-independent metrics")
print("-" * 70)
print(f"ROC-AUC            : {test_roc_auc:.4f}")
print(f"PR-AUC             : {test_pr_auc:.4f}")

print("\nThreshold-dependent metrics")
print("-" * 70)
print(f"Accuracy           : {test_accuracy:.4f}")
print(f"Precision          : {test_precision:.4f}")
print(f"Recall             : {test_recall:.4f}")
print(f"F1-score           : {test_f1:.4f}")

print("\nConfusion matrix")
print("-" * 70)
print(f"True Negatives     : {tn:,}")
print(f"False Positives    : {fp:,}")
print(f"False Negatives    : {fn:,}")
print(f"True Positives     : {tp:,}")

print("\nClassification report")
print("-" * 70)

print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["Normal", "Anomaly"],
        digits=4,
        zero_division=0
    )
)

# Final LSTM Anomaly Detection — Summary

## Objective

An LSTM-based binary classifier was developed to detect anomalies in the CIESOL sensor data.

The model predicts:

* `0` → Normal
* `1` → Anomaly

Each prediction is based on a sequence of **20 consecutive observations**.

---

## Data and preprocessing

* Chronological split:

  * **Train:** 60 days
  * **Validation:** 10 days
  * **Test:** 10 days
* Anomalies were injected into **train, validation, and test** so that all datasets contained labels.
* Synthetic anomaly targets: `PCO2EXT`, `PHEXT`, `PRAD`, `PRGINT`, `PTEXT`, `PVV`, `XCO2I`, `XHINV`, `XTINV`
* `UVENT_cen` and `UVENT_lN` were retained as contextual features.
* `XTS` was excluded from synthetic anomaly injection.
* Missing values were handled before sequence creation.
* Scaling was fitted on the **training data only** and then applied to validation and test data.

---

## Features

The final LSTM uses **19 features**:

* 12 sensor variables
* `hour_sin`
* `hour_cos`
* `is_night`
* `delta_temp`
* `delta_hum`
* `co2_gradient`
* `any_sensor_was_missing`

Sequence shape:

```text
(20 observations, 19 features)
```

Final sequence counts:

| Dataset    | Sequences |
| ---------- | --------: |
| Train      |   172,854 |
| Validation |    28,781 |
| Test       |    31,661 |

---

## LSTM Architecture

```text
Input: (20, 19)
      ↓
LSTM(64, return_sequences=True)
      ↓
Dropout(0.20)
      ↓
LSTM(32)
      ↓
Dropout(0.20)
      ↓
Dense(16, ReLU)
      ↓
Dense(1, Sigmoid)
```

**Total trainable parameters:** 34,465

Training configuration:

* Optimizer: Adam
* Initial learning rate: `0.001`
* Loss: Binary Cross-Entropy
* Batch size: `256`
* Maximum epochs: `50`
* Class weighting: applied to compensate for class imbalance
* Early stopping: based on validation PR-AUC
* Model checkpointing: based on validation PR-AUC

---

## Model Selection

The best validation PR-AUC was obtained at **epoch 6**:

* Validation PR-AUC: **0.9265**
* Validation ROC-AUC: **0.9533**

After epoch 6, training performance continued to improve while validation performance decreased, indicating overfitting.

Therefore, the model from the best validation checkpoint was retained:

```text
models/lstm_classifier_best.keras
```

---

## Threshold Selection

The anomaly decision threshold was selected using the **validation set only**.

A fine search between `0.20` and `0.30` was performed using F1-score.

Selected threshold:

```text
0.24
```

Validation performance at this threshold:

| Metric    | Validation |
| --------- | ---------: |
| Precision |     0.9830 |
| Recall    |     0.8425 |
| F1-score  |     0.9073 |
| TN        |     23,982 |
| FP        |         69 |
| FN        |        745 |
| TP        |      3,985 |

The threshold was then **frozen before evaluating the test set**.

---

## Final Test Results

The final model was evaluated on the unseen test set using the frozen threshold of `0.24`.

| Metric        |       Test |
| ------------- | ---------: |
| **ROC-AUC**   | **0.9688** |
| **PR-AUC**    | **0.9410** |
| **Accuracy**  | **0.9792** |
| **Precision** | **0.9645** |
| **Recall**    | **0.8828** |
| **F1-score**  | **0.9218** |

### Confusion Matrix

|                    | Predicted Normal | Predicted Anomaly |
| ------------------ | ---------------: | ----------------: |
| **Actual Normal**  |           27,123 |               143 |
| **Actual Anomaly** |              515 |             3,880 |

Test set:

* Normal sequences: **27,266**
* Anomaly sequences: **4,395**
* Total sequences: **31,661**

---

## Final Pipeline

```text
Raw sensor data
      ↓
Preprocessing
      ↓
Chronological train / validation / test split
      ↓
Anomaly injection
      ↓
Feature engineering
      ↓
Missing-value handling
      ↓
Training-based scaling
      ↓
20-observation sequence creation
      ↓
LSTM training
      ↓
Best model selected using validation PR-AUC
      ↓
Threshold selection using validation data
      ↓
Threshold frozen at 0.24
      ↓
Final evaluation on unseen test data
```

## Final Result

The final LSTM anomaly detector achieved:

**PR-AUC = 0.9410**
**ROC-AUC = 0.9688**
**Precision = 0.9645**
**Recall = 0.8828**
**F1-score = 0.9218**

The test set was used only for the final evaluation after model selection and threshold selection were completed.


In [ ]:

# ==============================================================
# STEP 12 — LSTM INFERENCE LATENCY
# ==============================================================

import time
import numpy as np
import pandas as pd

# Number of individual sequences to benchmark
N_BENCHMARK = min(1000, len(X_test))

# Use individual sequences so that latency represents
# one prediction rather than a large batch prediction.
X_benchmark = X_test[:N_BENCHMARK]

print("=" * 70)
print("LSTM INFERENCE LATENCY")
print("=" * 70)

print(f"Sequences benchmarked: {N_BENCHMARK:,}")

# --------------------------------------------------------------
# Warm-up
# --------------------------------------------------------------
# The first prediction can be slower because TensorFlow may
# initialize internal execution structures.

_ = best_model.predict(
    X_benchmark[:1],
    verbose=0
)

# --------------------------------------------------------------
# Measure individual prediction latency
# --------------------------------------------------------------

latencies_ms = []

for i in range(N_BENCHMARK):

    start_time = time.perf_counter()

    _ = best_model.predict(
        X_benchmark[i:i+1],
        verbose=0
    )

    end_time = time.perf_counter()

    latency_ms = (end_time - start_time) * 1000
    latencies_ms.append(latency_ms)

latencies_ms = np.array(latencies_ms)

# --------------------------------------------------------------
# Statistics
# --------------------------------------------------------------

mean_latency = np.mean(latencies_ms)
median_latency = np.median(latencies_ms)
min_latency = np.min(latencies_ms)
max_latency = np.max(latencies_ms)
p95_latency = np.percentile(latencies_ms, 95)
p99_latency = np.percentile(latencies_ms, 99)

# --------------------------------------------------------------
# Results
# --------------------------------------------------------------

print("\n" + "-" * 70)
print("LATENCY RESULTS")
print("-" * 70)

print(f"Mean latency       : {mean_latency:.3f} ms")
print(f"Median latency     : {median_latency:.3f} ms")
print(f"Minimum latency    : {min_latency:.3f} ms")
print(f"Maximum latency    : {max_latency:.3f} ms")
print(f"95th percentile    : {p95_latency:.3f} ms")
print(f"99th percentile    : {p99_latency:.3f} ms")

print("\n" + "-" * 70)
print("THROUGHPUT")
print("-" * 70)

mean_latency_seconds = mean_latency / 1000

if mean_latency_seconds > 0:
    predictions_per_second = 1 / mean_latency_seconds
else:
    predictions_per_second = np.inf

print(f"Predictions/second : {predictions_per_second:.2f}")

print("\n✓ Latency benchmark completed.")


In [ ]:
# ==============================================================
# STEP 12 — LSTM INFERENCE LATENCY (OPTIMIZED)
# ==============================================================

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "lstm_classifier_best.keras"
MODEL_PATH = "../models/lstm_classifier_best.keras"

# Load the best model selected during training
best_model = load_model(MODEL_PATH)

import time
import numpy as np
import pandas as pd

# Number of individual sequences to benchmark
N_BENCHMARK = min(1000, len(X_test))

# Use individual sequences so that latency represents
# one prediction rather than batch prediction.
X_benchmark = X_test[:N_BENCHMARK]

print("=" * 70)
print("LSTM INFERENCE LATENCY — OPTIMIZED")
print("=" * 70)

print(f"Sequences benchmarked: {N_BENCHMARK:,}")

# --------------------------------------------------------------
# Warm-up
# --------------------------------------------------------------
# Multiple warm-up predictions allow TensorFlow to initialize
# execution structures before timing starts.

N_WARMUP = 20

for i in range(N_WARMUP):
    _ = best_model(
        X_benchmark[i:i+1],
        training=False
    ).numpy()

print(f"Warm-up predictions  : {N_WARMUP}")

# --------------------------------------------------------------
# Measure individual prediction latency
# --------------------------------------------------------------
# Direct model(...) call is used instead of model.predict(...)
# to reduce Keras prediction overhead for single-sample inference.

latencies_ms = []

for i in range(N_BENCHMARK):

    start_time = time.perf_counter()

    prediction = best_model(
        X_benchmark[i:i+1],
        training=False
    )

    # Force TensorFlow to complete the computation
    # before stopping the timer.
    _ = prediction.numpy()

    end_time = time.perf_counter()

    latency_ms = (end_time - start_time) * 1000
    latencies_ms.append(latency_ms)

latencies_ms = np.asarray(latencies_ms)

# --------------------------------------------------------------
# Statistics
# --------------------------------------------------------------

mean_latency = np.mean(latencies_ms)
median_latency = np.median(latencies_ms)
min_latency = np.min(latencies_ms)
max_latency = np.max(latencies_ms)

p95_latency = np.percentile(latencies_ms, 95)
p99_latency = np.percentile(latencies_ms, 99)

# Standard deviation is useful to see how variable
# the inference latency is.
std_latency = np.std(latencies_ms)

# --------------------------------------------------------------
# Results
# --------------------------------------------------------------

print("\n" + "-" * 70)
print("LATENCY RESULTS")
print("-" * 70)

print(f"Mean latency       : {mean_latency:.3f} ms")
print(f"Median latency     : {median_latency:.3f} ms")
print(f"Minimum latency    : {min_latency:.3f} ms")
print(f"Maximum latency    : {max_latency:.3f} ms")
print(f"Standard deviation : {std_latency:.3f} ms")
print(f"95th percentile    : {p95_latency:.3f} ms")
print(f"99th percentile    : {p99_latency:.3f} ms")

# --------------------------------------------------------------
# Throughput
# --------------------------------------------------------------

print("\n" + "-" * 70)
print("THROUGHPUT")
print("-" * 70)

mean_latency_seconds = mean_latency / 1000

if mean_latency_seconds > 0:
    predictions_per_second = 1 / mean_latency_seconds
else:
    predictions_per_second = np.inf

print(f"Predictions/second : {predictions_per_second:.2f}")

# --------------------------------------------------------------
# Final interpretation
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)

print(
    f"Typical inference latency (median): "
    f"{median_latency:.2f} ms"
)

print(
    f"Average inference latency (mean): "
    f"{mean_latency:.2f} ms"
)

print(
    f"Worst observed latency (max): "
    f"{max_latency:.2f} ms"
)

print(
    f"95% of predictions completed within: "
    f"{p95_latency:.2f} ms"
)

print(
    f"99% of predictions completed within: "
    f"{p99_latency:.2f} ms" 
)

print("\n✓ Optimized latency benchmark completed.")

In [ ]:
# ==============================================================
# STEP 12 — OPTIMIZED LSTM INFERENCE LATENCY
#              TensorFlow Function + XLA
# ==============================================================

import time
import numpy as np
import tensorflow as tf

print("=" * 70)
print("OPTIMIZED LSTM INFERENCE LATENCY")
print("=" * 70)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs available     : {tf.config.list_physical_devices('GPU')}")
print(f"X_test shape       : {X_test.shape}")
print(f"X_test dtype       : {X_test.dtype}")

# --------------------------------------------------------------
# Prepare benchmark data
# --------------------------------------------------------------

N_BENCHMARK = min(2048, len(X_test))

X_benchmark = tf.convert_to_tensor(
    X_test[:N_BENCHMARK].astype(np.float32)
)

print(f"Sequences benchmarked: {N_BENCHMARK:,}")

# --------------------------------------------------------------
# Compile inference function with XLA
# --------------------------------------------------------------

@tf.function(
    jit_compile=True,
    reduce_retracing=True
)
def optimized_inference(x):

    return best_model(
        x,
        training=False
    )

# --------------------------------------------------------------
# Warm-up / XLA compilation
# --------------------------------------------------------------

print("\nCompiling and warming up XLA...")

for i in range(5):
    _ = optimized_inference(
        X_benchmark[i:i+1]
    ).numpy()

print("✓ XLA inference ready")

# --------------------------------------------------------------
# Measure SINGLE-SEQUENCE latency
# --------------------------------------------------------------

latencies_ms = []

for i in range(N_BENCHMARK):

    x_single = X_benchmark[i:i+1]

    start_time = time.perf_counter()

    prediction = optimized_inference(x_single)

    # Force synchronous execution
    _ = prediction.numpy()

    end_time = time.perf_counter()

    latency_ms = (
        end_time - start_time
    ) * 1000

    latencies_ms.append(latency_ms)

latencies_ms = np.asarray(latencies_ms)

# --------------------------------------------------------------
# Statistics
# --------------------------------------------------------------

mean_latency = np.mean(latencies_ms)
median_latency = np.median(latencies_ms)
min_latency = np.min(latencies_ms)
max_latency = np.max(latencies_ms)

std_latency = np.std(latencies_ms)

p95_latency = np.percentile(
    latencies_ms,
    95
)

p99_latency = np.percentile(
    latencies_ms,
    99
)

# --------------------------------------------------------------
# Throughput
# --------------------------------------------------------------

predictions_per_second = (
    1000 / mean_latency
)

# --------------------------------------------------------------
# Results
# --------------------------------------------------------------

print("\n" + "-" * 70)
print("SINGLE-SEQUENCE LATENCY")
print("-" * 70)

print(f"Mean latency       : {mean_latency:.3f} ms")
print(f"Median latency     : {median_latency:.3f} ms")
print(f"Minimum latency    : {min_latency:.3f} ms")
print(f"Maximum latency    : {max_latency:.3f} ms")
print(f"Standard deviation : {std_latency:.3f} ms")
print(f"95th percentile    : {p95_latency:.3f} ms")
print(f"99th percentile    : {p99_latency:.3f} ms")

print("\n" + "-" * 70)
print("THROUGHPUT")
print("-" * 70)

print(
    f"Predictions/second : "
    f"{predictions_per_second:.2f}"
)

print("\n" + "=" * 70)
print("✓ XLA latency benchmark completed")
print("=" * 70)